In [0]:
""" Utiities for Ingestion, transformation and enrichment """

import logging
from datetime import datetime
import json

from pyspark.sql import functions as F
from pyspark.sql import types as T
from pyspark.sql import Window as W
from pyspark.sql import DataFrame
from pyspark.sql.types import StructType

# --- logger --- #
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)-8s | %(message)s",
    datefmt="%Y-%m-%d %H:%M:%S",
)
logger = logging.getLogger("ecommerce_pipeline")

In [0]:
def parse_config_from_json(json_path):
    """Read a JSON file and return its contents as a dict."""
    
    try:
        with open(json_path, "r") as f:
            return json.load(f)
    except FileNotFoundError:
        raise FileNotFoundError(f"JSON file not found: {json_path}")
    except json.JSONDecodeError as e:
        raise RuntimeError(f"Failed to parse JSON '{json_path}': {e}")
    except Exception as e:
        raise RuntimeError(f"Failed to read JSON '{json_path}': {e}")

In [0]:
# --- File Readers ---- #

def json_reader(spark, file_path: str, multi_line: bool = True) -> DataFrame:
    """ Read a JSON file into a DataFrame """
    try:
        logger.info("Reading JSON: %s (multi_line=%s)", file_path, multi_line)
        if multi_line:
            df = (
                spark.read.format("json")
                .option("multiLine", multi_line)
                .load(file_path)
            )
        else:
            df = spark.read.json(file_path)
        logger.info("JSON read complete – %d columns detected", len(df.columns))
        return df
    except Exception as e:
        if "Path does not exist" in str(e):
            raise FileNotFoundError(f"JSON file not found: {file_path}") from e
        raise RuntimeError(f"Failed to read JSON '{file_path}': {e}") from e


def csv_reader(spark, file_path: str, delimiter: str = ",", header: bool = True) -> DataFrame:
    """ Read a CSV file into a DataFrame """
    try:
        logger.info("Reading CSV: %s (delimiter='%s', header=%s)", file_path, delimiter, header)
        df = (
            spark.read.format("csv")
            .options(header=header, delimiter=delimiter, escape='"')
            .load(file_path)
        )
        logger.info("CSV read complete – %d columns detected", len(df.columns))
        return df
    except Exception as e:
        if "Path does not exist" in str(e):
            raise FileNotFoundError(f"CSV file not found: {file_path}") from e
        raise RuntimeError(f"Failed to read CSV '{file_path}': {e}") from e


def excel_reader(spark, file_path: str, sheet_name: str) -> DataFrame:
    """Read an Excel sheet into a DataFrame."""
    try:
        logger.info("Reading Excel: %s (sheet='%s')", file_path, sheet_name)
        df = (
            spark.read.format("excel")
            .option("headerRows", 1)
            .option("inferSchema", "true")
            .load(file_path, sheetName=sheet_name)
        )
        logger.info("Excel read complete – %d columns detected", len(df.columns))
        return df
    except Exception as e:
        if "Path does not exist" in str(e):
            raise FileNotFoundError(f"Excel file not found: {file_path}") from e
        raise RuntimeError(f"Failed to read Excel '{file_path}': {e}") from e

In [0]:
# ---------------------------------------------------------------------------
# Schema Helpers
# ---------------------------------------------------------------------------

def apply_schema(df: DataFrame, table_name: str, schema_dict: dict) -> DataFrame:
    """Rename columns according to a mapping dictionary.

    Parameters
    ----------
    df : DataFrame         – Input DataFrame.
    table_name : str       – Key into *schema_dict*.
    schema_dict : dict     – ``{table_name: {original_col: new_col, ...}}``.

    Returns
    -------
    DataFrame with renamed columns.  Columns not in the mapping are kept
    with their original names.

    Raises
    ------
    KeyError – if *table_name* is not found in *schema_dict*.
    """
    if table_name not in schema_dict:
        raise KeyError(
            f"Alias key '{table_name}' not found in schema_dict. "
            f"Available keys: {list(schema_dict.keys())}"
        )
    aliases = schema_dict[table_name]
    logger.info("Applying %d column aliases for '%s'", len(aliases), table_name)
    return df.select(
        [df[col].alias(aliases[col]) if col in aliases else df[col] for col in df.columns]
    )

In [0]:
# ---------------------------------------------------------------------------
# Delta Writer
# ---------------------------------------------------------------------------

def delta_writer(
    df: DataFrame,
    table_name: str,
    mode: str = "overwrite",
    partition_by: list = None,
) -> None:
    """Write a DataFrame to a Delta table in Unity Catalog.

    The function uses ``overwriteSchema`` instead of ``DROP TABLE`` for
    atomicity – readers always see a consistent table.

    Parameters
    ----------
    df : DataFrame         – Data to write.
    table_name : str       – Fully-qualified table name (catalog.schema.table).
    mode : str             – Spark write mode (default 'overwrite').
    partition_by : list    – Optional list of columns to partition by.

    Raises
    ------
    ValueError – if *df* is None or *table_name* is empty.
    RuntimeError – on write failure.
    """
    if df is None:
        raise ValueError("Cannot write a None DataFrame.")
    if not table_name or not table_name.strip():
        raise ValueError("table_name must be a non-empty string.")

    try:
        logger.info("Writing to %s (mode=%s)", table_name, mode)
        writer = (
            df.write
            .mode(mode)
            .option("overwriteSchema", "true")
            .format("delta")
        )
        if partition_by:
            writer = writer.partitionBy(*partition_by)
        writer.saveAsTable(table_name)
        logger.info("Write complete: %s", table_name)
    except Exception as e:
        raise RuntimeError(f"Failed to write Delta table '{table_name}': {e}") from e

In [0]:
# ---------------------------------------------------------------------------
# Ingestion Orchestration
# ---------------------------------------------------------------------------

_READERS = {
    "csv": csv_reader,
    "json": json_reader,
    "excel": excel_reader,
}

_REQUIRED_CONFIG_KEYS = {"file_path", "file_format", "target_table"}


def ingest_data(
    spark,
    file_path: str,
    file_format: str,
    target_table: str,
    alias_key: str = None,
    column_alias_dict: dict = None,
    reader_options: dict = None,
) -> DataFrame:
    """End-to-end ingestion: read source file → apply column aliases → write Delta table.

    Parameters
    ----------
    spark : SparkSession
    file_path : str            – Volume path to the source file.
    file_format : str          – One of 'csv', 'json', 'excel'.
    target_table : str         – Fully-qualified Delta table name.
    alias_key : str, optional  – Key into *column_alias_dict* for renaming columns.
    column_alias_dict : dict   – ``{alias_key: {orig_col: new_col}}``.
    reader_options : dict      – Extra kwargs forwarded to the reader function.

    Returns
    -------
    DataFrame – The ingested (and optionally aliased) DataFrame.
    """
    reader_fn = _READERS.get(file_format)
    if reader_fn is None:
        raise ValueError(
            f"Unsupported file format '{file_format}'. "
            f"Supported: {list(_READERS.keys())}"
        )

    opts = reader_options or {}
    df = reader_fn(spark, file_path, **opts)

    if alias_key and column_alias_dict:
        df = apply_schema(df, alias_key, column_alias_dict)

    delta_writer(df, target_table)
    return df


def _validate_config(config: dict, index: int) -> None:
    """Validate a single ingestion config dict."""
    missing = _REQUIRED_CONFIG_KEYS - config.keys()
    if missing:
        raise ValueError(
            f"Config at index {index} is missing required keys: {missing}"
        )


def run_ingestion(
    spark,
    ingestion_configs: list,
    column_alias_dict: dict = None,
) -> dict:
    """Batch-process a list of ingestion configs.

    Parameters
    ----------
    spark : SparkSession
    ingestion_configs : list[dict] – Each dict must contain
        ``file_path``, ``file_format``, ``target_table`` and optionally
        ``alias_key`` and ``reader_options``.
    column_alias_dict : dict, optional – Shared alias dictionary.

    Returns
    -------
    dict – ``{target_table: DataFrame}`` for every successfully ingested table.
           Failed tables are logged but do not halt the batch.
    """
    if not ingestion_configs:
        logger.warning("run_ingestion called with an empty config list.")
        return {}

    # Validate all configs upfront
    for idx, cfg in enumerate(ingestion_configs):
        _validate_config(cfg, idx)

    results = {}
    errors = []
    for config in ingestion_configs:
        table = config["target_table"]
        try:
            logger.info("Ingesting %s → %s", config["file_path"], table)
            df = ingest_data(
                spark,
                file_path=config["file_path"],
                file_format=config["file_format"],
                target_table=table,
                alias_key=config.get("alias_key"),
                column_alias_dict=column_alias_dict,
                reader_options=config.get("reader_options"),
            )
            row_count = df.count()
            results[table] = df
            logger.info("✅ %s – %d rows ingested", table, row_count)
        except Exception as e:
            errors.append((table, str(e)))
            logger.error("❌ %s – %s", table, e)

    if errors:
        logger.warning("%d of %d ingestion tasks failed.", len(errors), len(ingestion_configs))
    return results

In [0]:
# ---------------------------------------------------------------------------
# Data Quality Validation
# ---------------------------------------------------------------------------

class DataQualityError(Exception):
    """Raised when a data-quality check fails."""
    pass


def validate_not_null(df: DataFrame, columns: list, label: str = "") -> dict:
    """Assert that specified columns contain zero NULLs.

    Parameters
    ----------
    df : DataFrame
    columns : list[str] – Column names to check.
    label : str         – Optional label for log messages.

    Returns
    -------
    dict – ``{column_name: null_count}``  (only columns with nulls > 0).

    Raises
    ------
    DataQualityError – if any column has NULLs.
    """
    missing_cols = [c for c in columns if c not in df.columns]
    if missing_cols:
        raise DataQualityError(
            f"[{label}] Columns not found in DataFrame: {missing_cols}"
        )

    null_counts = {}
    for col in columns:
        cnt = df.filter(F.col(col).isNull()).count()
        if cnt > 0:
            null_counts[col] = cnt

    if null_counts:
        raise DataQualityError(
            f"[{label}] NULL values detected: {null_counts}"
        )
    logger.info("[%s] validate_not_null PASSED for %s", label, columns)
    return null_counts


def validate_no_duplicates(
    df: DataFrame, key_columns: list, label: str = ""
) -> int:
    """Assert that there are no duplicate rows based on key columns.

    Returns
    -------
    int – Number of duplicate rows (0 on success).

    Raises
    ------
    DataQualityError – if duplicates exist.
    """
    total = df.count()
    distinct = df.select(key_columns).distinct().count()
    dupes = total - distinct

    if dupes > 0:
        raise DataQualityError(
            f"[{label}] {dupes} duplicate rows on key(s) {key_columns}"
        )
    logger.info("[%s] validate_no_duplicates PASSED (keys=%s)", label, key_columns)
    return dupes


def validate_schema(
    df: DataFrame, expected_schema: StructType, label: str = ""
) -> bool:
    """Validate that a DataFrame matches the expected StructType schema.

    Checks column names and data types.  Extra columns in the DataFrame are
    allowed; only *missing* or *mistyped* expected columns raise an error.

    Returns
    -------
    bool – True on success.

    Raises
    ------
    DataQualityError – on mismatch.
    """
    actual_fields = {f.name: f.dataType for f in df.schema.fields}
    issues = []
    for field in expected_schema.fields:
        if field.name not in actual_fields:
            issues.append(f"Missing column: '{field.name}'")
        elif actual_fields[field.name] != field.dataType:
            issues.append(
                f"Type mismatch for '{field.name}': "
                f"expected {field.dataType}, got {actual_fields[field.name]}"
            )
    if issues:
        raise DataQualityError(
            f"[{label}] Schema validation failed:\n  " + "\n  ".join(issues)
        )
    logger.info("[%s] validate_schema PASSED", label)
    return True


def validate_row_count(
    df: DataFrame,
    min_rows: int = 1,
    max_rows: int = None,
    label: str = "",
) -> int:
    """Assert row count is within [min_rows, max_rows].

    Returns
    -------
    int – Actual row count.

    Raises
    ------
    DataQualityError – if count is outside the range.
    """
    count = df.count()
    if count < min_rows:
        raise DataQualityError(
            f"[{label}] Row count {count} < minimum {min_rows}"
        )
    if max_rows is not None and count > max_rows:
        raise DataQualityError(
            f"[{label}] Row count {count} > maximum {max_rows}"
        )
    logger.info("[%s] validate_row_count PASSED (%d rows)", label, count)
    return count


def run_quality_checks(df: DataFrame, checks: list, label: str = "") -> dict:
    """Run multiple data-quality checks in one call.

    Parameters
    ----------
    df : DataFrame
    checks : list[dict] – Each dict describes one check::

        {"check": "not_null",       "columns": ["id", "name"]}
        {"check": "no_duplicates",  "key_columns": ["id"]}
        {"check": "schema",         "expected_schema": StructType(...)}
        {"check": "row_count",      "min_rows": 1, "max_rows": 100000}

    label : str – Prefix for log messages.

    Returns
    -------
    dict – ``{check_name: 'PASSED' | error_message}``.
    """
    results = {}
    dispatch = {
        "not_null":      lambda c: validate_not_null(df, c["columns"], label),
        "no_duplicates": lambda c: validate_no_duplicates(df, c["key_columns"], label),
        "schema":        lambda c: validate_schema(df, c["expected_schema"], label),
        "row_count":     lambda c: validate_row_count(df, c.get("min_rows", 1), c.get("max_rows"), label),
    }
    for chk in checks:
        name = chk["check"]
        fn = dispatch.get(name)
        if fn is None:
            results[name] = f"Unknown check type: '{name}'"
            continue
        try:
            fn(chk)
            results[name] = "PASSED"
        except DataQualityError as e:
            results[name] = str(e)
            logger.warning("Quality check '%s' FAILED: %s", name, e)
    return results

In [0]:
# ---------------------------------------------------------------------------
# Reusable Transformation Helpers
# ---------------------------------------------------------------------------

def clean_dataframe(
    df: DataFrame,
    primary_key: str,
    fill_defaults: dict = None,
    target_schema: StructType = None,
    date_columns: dict = None,
) -> DataFrame:
    """Standard cleaning pipeline: filter → fill → deduplicate → cast.

    This function consolidates the repeated pattern used across the
    transformation notebook into a single reusable call.

    Parameters
    ----------
    df : DataFrame            – Raw input DataFrame.
    primary_key : str         – Column that must be non-NULL.
    fill_defaults : dict      – ``{column: default_value}`` for ``na.fill``.
    target_schema : StructType– Desired output schema (casts each column).
    date_columns : dict       – ``{column_name: date_format}`` for columns
                                that need ``to_date`` conversion instead of
                                a simple cast.

    Returns
    -------
    DataFrame – Cleaned, deduplicated, and casted DataFrame.
    """
    if primary_key not in df.columns:
        raise ValueError(f"Primary key '{primary_key}' not in DataFrame columns: {df.columns}")

    # 1. Filter out NULLs on primary key
    logger.info("Cleaning: filtering NULLs on '%s'", primary_key)
    df = df.filter(F.col(primary_key).isNotNull())

    # 2. Fill defaults
    if fill_defaults:
        logger.info("Cleaning: filling %d default values", len(fill_defaults))
        df = df.na.fill(fill_defaults)

    # 3. Deduplicate
    before = df.count()
    df = df.dropDuplicates()
    after = df.count()
    logger.info("Cleaning: dedup removed %d rows (%d → %d)", before - after, before, after)

    # 4. Cast to target schema
    if target_schema:
        date_cols = date_columns or {}
        select_exprs = []
        for field in target_schema.fields:
            if field.name in date_cols:
                select_exprs.append(
                    F.to_date(F.col(field.name), date_cols[field.name]).alias(field.name)
                )
            else:
                select_exprs.append(F.col(field.name).cast(field.dataType))
        df = df.select(*select_exprs)
        logger.info("Cleaning: schema cast applied (%d columns)", len(target_schema.fields))

    return df